# KVQuant Implementation -- full-precision baseline (GSM8K + ARC-Challenge + HellaSwag)

Full-precision baseline over three datasets: GSM8K (generative, extract the
final number), ARC-Challenge, and HellaSwag (both multiple-choice, scored
generate-and-extract: the model generates and we parse the chosen letter).

All three datasets share the SAME real `generate()`-based measurement path,
so TTFT/TBT/latency, KV-cache memory (measured from the real cache tensors
this model produces), and per-answer perplexity are directly comparable
across datasets. This is deliberate: because the point of the notebook is to
measure generation timing and KV-cache memory, the MCQ datasets use
generate-and-extract rather than the leaderboard log-likelihood (`acc_norm`)
protocol -- a single teacher-forced forward pass would not exercise the
decode-time KV cache the way generation does. Consequently the MCQ accuracies
here are NOT directly comparable to Open-LLM-Leaderboard HellaSwag/ARC numbers.

Few-shot prefixes for ARC and HellaSwag are NOT hand-written. Following the
lm-eval-harness convention, exemplars are drawn by a seeded `random.Random`
sample from each dataset's own train split (non-CoT, direct-answer format:
question + lettered options + `Answer: <letter>`). The eval pool for each
dataset is the *remainder* of its train split (train minus the sampled
few-shot exemplars) combined with its full test (ARC) / validation
(HellaSwag) split; the reproducible random eval subset is drawn from that
combined pool. No exemplar ever also appears as an eval question, so there is
no leakage. ARC uses 25-shot and HellaSwag 10-shot by default (the
leaderboard conventions); both are configurable constants. HellaSwag
context/endings get the standard harness text cleanup before formatting.

**Measurement methodology (identical structure to H2O/KVQuant):** generation uses a hand-rolled prefill + greedy-decode loop under eager attention, not `model.generate()` with SDPA -- see the model-loading and machinery cells in Setup/Helper Functions. This removes a latency confound from earlier versions of this notebook, which ran `model.generate()` under SDPA's fused kernels while H2O and KVQuant both pay an unavoidable eager-attention cost; eager and SDPA compute the identical attention math, so this changes latency only, not accuracy/perplexity. A VALIDATION cell right after `generate_gsm8k` is defined confirms the new hand-rolled loop reproduces plain `model.generate()`'s predicted answer before any timing number is trusted.

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [1]:
!hostname

DESKTOP-32E0L5J


In [2]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods (kept
# identical to the 2-bit/3-bit/4-bit notebooks even though this one never
# clones/patches the KVCacheCompression repo, so there's no risk of a
# transformers/tokenizers/etc. version mismatch skewing the comparison).

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.20.3" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

# Patch - transformers==4.43.4 hard-enforces "tokenizers>=0.19,<0.20" at
# IMPORT time (transformers/dependency_versions_check.py), not just at
# pip-install time -- so even though tokenizers==0.20.3 installs fine
# (needed since 0.19.x has no Python 3.13 wheel), "import transformers"
# still raises ImportError unless this hardcoded constraint is relaxed on
# disk first. Patched via a direct file edit -- importing transformers to
# patch it in-memory is not an option, since that import is exactly what
# triggers the failing check. Safe to run even on the old (working)
# environment: on Python <3.13 this is a no-op the moment the constraint
# line has already been relaxed, and re-running it is idempotent.
import importlib.util
import re

_transformers_spec = importlib.util.find_spec("transformers")
_deps_table_path = os.path.join(os.path.dirname(_transformers_spec.origin), "dependency_versions_table.py")

with open(_deps_table_path, "r") as f:
    _deps_content = f.read()

_deps_content_new, _n_subs = re.subn(
    r'("tokenizers":\s*)"tokenizers>=0\.19,<0\.20"',
    r'\1"tokenizers>=0.19,<0.21"',
    _deps_content,
)
if _n_subs > 0:
    with open(_deps_table_path, "w") as f:
        f.write(_deps_content_new)
    print(f"Patched {_deps_table_path}: tokenizers constraint relaxed to <0.21 "
          "(allows tokenizers==0.20.3 -- 0.19.x has no Python 3.13 wheel).")
else:
    print(f"NOTE: tokenizers constraint in {_deps_table_path} was not the expected "
          "'>=0.19,<0.20' (already patched, or transformers version differs) -- "
          "no change made.")

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import os
import re
import shutil
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) for consistency with the KVQuant-family notebooks,
# where the Fisher calibration cell needs them available early in Setup.
# This notebook has no such dependency itself (no calibration step), but
# keeping the split identical across the whole notebook family avoids
# Helper Functions containing different things in different notebooks.
# sync_if_cuda/clear_memory have no early dependency anywhere and live in
# Helper Functions with the rest of the genuinely cross-dataset machinery.

In [ ]:
# Block 3 - Experiment settings.
# Sampling policy for GSM8K (the only dataset in this notebook): up to
# 1,024 random valid questions, deterministic seed 42. Sampling happens
# after validity filtering; selected indices are sorted back into source
# order so the subset is random while evaluation order stays stable. No
# ABITS/SPARSITY_THRESHOLD/quantizer settings here -- this notebook never
# quantizes anything.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if os.path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
QA_EVAL_SAMPLES = 2048
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_baseline_full_precision"

random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Random sampling seed:", SHARED_SEED)
print("GSM8K random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)

# NOTE: ARC-Challenge/HellaSwag no longer take few-shot exemplars or a
# capped generation length -- they are scored zero-shot via teacher-forced
# per-choice likelihood (see the Shared multiple-choice (MC) scoring
# machinery cell, below), matching large_sample_implementations exactly.
# GSM8K is unaffected: it keeps GSM8K_FEWSHOT_PREFIX above.


In [ ]:
# Block - Load tokenizer + the untouched full-precision model. No repo
# clone, no patches, no Fisher calibration, no Quantize step -- none of
# that machinery is needed for a model that's never quantized. This is the
# exact same loading code the combined v3 notebook used for model_fp, with
# ONE change: attn_implementation is "eager", not "sdpa". This notebook's
# generation loop (below) is now a hand-rolled manual prefill/decode loop
# identical in structure to the H2O/KVQuant notebooks', not a plain
# model.generate() call -- running it under SDPA's fused kernels while
# H2O/KVQuant pay the "eager tax" would make every latency comparison
# across this notebook family unfair. Eager attention is mathematically
# identical to SDPA (the same exact softmax attention, just a different,
# unfused kernel), so this changes latency only, never accuracy/perplexity.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "eager",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print("Loading full-precision baseline model...")
model_fp = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_fp = model_fp.to(DEVICE)
model_fp.eval()
model_fp.config.use_cache = True

device = next(model_fp.parameters()).device
print("model_fp: full-precision baseline (eager attention), loaded and ready.")


## Helper Functions

Shared inference machinery used across all three datasets (GSM8K,
ARC-Challenge, HellaSwag).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - Hand-rolled eager decode machinery -- the full-precision
# baseline's analogue of the H2O/KVQuant notebooks' engine plumbing
# (make_h2o_engines/h2o_prefill/h2o_step, real_reset/real_prefill/
# real_step). There is no eviction or quantization decision to make here --
# this cell exists ONLY so the baseline's generation loop has the exact
# same manual prefill + per-token decode STRUCTURE as the other two
# notebooks (explicit position_ids/attention_mask per step, TTFT
# timestamped right after the prefill forward, one forward call per
# generated token), instead of hiding that structure inside
# model.generate(). Running the identical loop shape under identical
# (eager) attention is what makes this notebook's TTFT/TBT/latency numbers
# comparable to H2O's and KVQuant's -- previously this notebook used
# model.generate() with fused SDPA kernels, a strictly easier case than
# what the other two notebooks pay for.


def cache_to_legacy(past_key_values):
    if past_key_values is None:
        return None
    if isinstance(past_key_values, tuple):
        return past_key_values
    if hasattr(past_key_values, "to_legacy_cache"):
        return past_key_values.to_legacy_cache()
    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


def get_cache_tokens(past_key_values):
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    return int(past_key_values[0][0].shape[2])


def dense_kv_cache_bytes(past_key_values):
    """Real bytes of the tensors actually resident in past_key_values right
    now, at their current dtype -- the baseline never evicts or quantizes,
    so this is simply the full dense cache."""
    past_key_values = cache_to_legacy(past_key_values)
    if past_key_values is None:
        return 0
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return int(total)


@torch.no_grad()
def baseline_prefill(input_ids):
    """One batched forward over the whole prompt -- returns (last_logits,
    past_key_values), matching h2o_prefill's/real_prefill's role."""
    outputs = model_fp(input_ids=input_ids, use_cache=True, return_dict=True)
    return outputs.logits[:, -1, :], outputs.past_key_values


@torch.no_grad()
def baseline_step(token_tensor, abs_pos, past_key_values):
    """One decode step. Explicit position_ids/attention_mask (rather than
    relying on model.generate()'s internal bookkeeping) so this loop has
    the exact same shape as h2o_step/real_step -- harmless here since the
    baseline cache is never pruned or resized, but it keeps all three
    notebooks' decode-step mechanics structurally identical."""
    cache_len_before = get_cache_tokens(past_key_values)
    attention_mask = torch.ones((1, cache_len_before + 1), dtype=torch.long, device=device)
    position_ids = torch.tensor([[abs_pos]], dtype=torch.long, device=device)
    outputs = model_fp(
        input_ids=token_tensor,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        position_ids=position_ids,
        use_cache=True,
        return_dict=True,
    )
    return outputs.logits[:, -1, :], outputs.past_key_values


In [ ]:
def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)

    sample_count = min(int(max_samples), len(items))

    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )

    return [items[index] for index in selected_indices], selected_indices

In [ ]:
def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err

In [ ]:
# ============================================================================
# Shared multiple-choice (MC) scoring machinery -- ARC-Challenge and
# HellaSwag are scored via teacher-forced per-choice likelihood, matching the
# large_sample_implementations family's methodology exactly (not
# generate-and-extract): every answer choice is teacher-forced token-by-token
# through the SAME hand-rolled decode step used everywhere else in this
# notebook (baseline_step), and its per-token cross-entropy is summed into
# that choice's NLL. The model is never asked to generate anything for these
# two datasets -- there is no few-shot prefix and no chain-of-thought either;
# both datasets are zero-shot. GSM8K is untouched -- it keeps its own
# separate generative grading (generate_gsm8k, above).
#
# lm_eval_encode_pair(context, choice)
#     Tokenizes context+continuation JOINTLY then splits at the context's own
#     token count (matching lm-evaluation-harness) -- preserves true
#     tokenizer boundary behavior, which separately tokenizing context and
#     continuation can miss (BPE merges can span the boundary).
#
# score_mc_choice_baseline(prompt, choice)
#     Walks the full (context + choice) sequence one token at a time via
#     baseline_step (the exact same decode-step function GSM8K's hand-rolled
#     loop uses), scoring every position from the choice's first token
#     onward. TTFT = first step's time; TBT = mean of the remaining steps;
#     total latency = sum of every step. Memory is the real dense cache size
#     at the end of the walk (the cache only grows, so end == peak).
#
# score_mc_question_baseline(prompt, choices, gold_index)
#     Scores every choice via score_mc_choice_baseline, then reports BOTH
#     raw (argmin of un-normalized NLL) and normalized (argmin of NLL /
#     choice's own character length -- the standard lm-eval-harness acc_norm
#     convention) accuracy. Perplexity comes ONLY from the gold choice's own
#     mean NLL. TTFT = mean across choices; TBT = weighted mean across every
#     choice's decode steps; total latency = SUM across choices (the real
#     cost of answering the question is the cost of scoring every choice);
#     peak memory = MAX across choices.
# ============================================================================


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")

    return context_ids, continuation_ids

@torch.no_grad()
def score_mc_choice_baseline(prompt, choice):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        step_logits, past_key_values = baseline_step(step_input, pos, past_key_values)
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = dense_kv_cache_bytes(past_key_values)

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_baseline(prompt, choices, gold_index):
    choice_results = [score_mc_choice_baseline(prompt, choice) for choice in choices]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }


## GSM8K

In [ ]:
# Block - GSM8K loading: combine all splits, then random sampling.
# Load train + test splits, build the complete list of valid question/answer
# pairs, then select up to 1,024 examples with seed 42. This creates a
# reproducible random sample across the entire GSM8K dataset.

def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None

gsm8k_train = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="train",
    desc="GSM8K train load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

gsm8k_test = robust_call(
    load_dataset,
    "gsm8k",
    "main",
    split="test",
    desc="GSM8K test load",
    on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

# Combine all official GSM8K splits
gsm8k_all = list(gsm8k_train) + list(gsm8k_test)
all_gsm8k_pairs = []

for item in gsm8k_all:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)

print(
    "GSM8K selected valid-item indices (first 20):",
    gsm8k_selected_indices[:20]
)

In [ ]:
# Block - GSM8K generation with the full-precision baseline, via a
# hand-rolled prefill + greedy-decode loop (baseline_prefill/baseline_step)
# identical in structure to the H2O/KVQuant notebooks' -- NOT a plain
# model.generate() call. This replaces the previous version's
# model.generate() + StoppingCriteria implementation: that version ran
# under SDPA's fused kernels and measured TTFT/TBT via a callback fired
# from inside generate()'s C-level loop, a fundamentally different (and
# structurally easier/faster) code path than what H2O and KVQuant pay for.
# Running the SAME loop shape under the SAME (eager) attention as the other
# two notebooks is what makes this notebook's latency numbers comparable to
# theirs. See the VALIDATION cell right after this one for a check that the
# new loop reproduces the old model.generate() path's accuracy and
# generated text before trusting any latency number from here on.
#
# Timing definitions (identical to the whole family):
#   TTFT  = time from generation start until the first generated token's
#           logits are ready (the batched prefill forward).
#   total = wall time of the whole generation.
#   TBT   = (total - TTFT) / max(n_generated - 1, 1).
# n_generated counts every token an actual forward call produced,
# INCLUDING the final EOS-producing step -- tracked as n_forward_tokens,
# while gen_text excludes EOS itself.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same hand-rolled decode loop -- no separate teacher-forced pass.
# Per-step logits (last_logits) are already computed for greedy decoding
# regardless, so detecting the answer span costs one extra cheap text
# decode per token; the actual log-probability scoring is deferred until
# after gen_end, using logits saved during the loop. The span starts right
# after four consecutive '#' characters (the "####" marker GSM8K answers
# use) and stops the moment another '#' appears. If "####" never appears,
# perplexity is None for that question.
#
# Answer grading is VERBATIM from the family: truncate the generation at
# the first "Question:", extract "#### <num>" (falling back to the last
# number), compare |pred - gold| < 1e-4.
#
# Memory: an untimed pass over the full (prompt + generated) sequence after
# gen_end, purely to measure the dense cache's real byte size -- the
# baseline never evicts or quantizes, so this IS the peak.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


@torch.no_grad()
def generate_gsm8k(question):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    sync_if_cuda()
    gen_start = time.perf_counter()

    # ---- Prefill: one batched forward ----
    last_logits, pkv = baseline_prefill(enc["input_ids"])
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    generated_ids = []
    n_forward_tokens = 1  # the prefill forward already produced next_id

    # ---- Live perplexity-span detection state -- see header comment ----
    hash_streak = 0
    in_answer_span = False
    answer_done = False
    answer_token_ids = []
    answer_logits = []

    # ---- Greedy decode ----
    for step in range(GSM8K_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)

        if not answer_done:
            tok_text = tokenizer.decode([next_id])
            if in_answer_span:
                if "#" in tok_text:
                    answer_done = True
                else:
                    answer_token_ids.append(next_id)
                    answer_logits.append(last_logits.detach())
            else:
                for ch in tok_text:
                    hash_streak = hash_streak + 1 if ch == "#" else 0
                if hash_streak >= 4:
                    in_answer_span = True
        if "\n" in tokenizer.decode([next_id]):        # early stop: halt after the #### answer line
            _run_txt = tokenizer.decode(generated_ids)
            if "####" in _run_txt and "\n" in _run_txt.split("####", 1)[-1]:
                break

        if step == GSM8K_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = baseline_step(token_tensor, abs_pos, pkv)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    # Perplexity of the model's own predicted final number, scored here
    # (fully after gen_end, untimed) from the logits saved above.
    if answer_token_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(answer_token_ids, answer_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(answer_token_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Untimed pass over the full generated sequence, purely to measure
    # memory. Does not affect the timed loop above.
    if generated_ids:
        full_ids = torch.cat(
            [enc["input_ids"], torch.tensor([generated_ids], dtype=torch.long, device=device)], dim=1
        )
    else:
        full_ids = enc["input_ids"]
    _, final_pkv = baseline_prefill(full_ids)
    peak_bytes = dense_kv_cache_bytes(final_pkv)

    return {
        "full_prompt": prompt, "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


In [ ]:
# Block - VALIDATION: hand-rolled eager loop vs. plain model.generate(),
# on the SAME (eager) model_fp -- run this before trusting any
# latency/accuracy number from this notebook. It answers one question only:
# does the new manual prefill/decode loop (baseline_prefill/baseline_step,
# used everywhere below) reproduce the SAME generation as a normal
# model.generate() call, i.e. is its position_ids/attention_mask/cache
# bookkeeping correct?
#
# This is NOT a check of eager-vs-sdpa numerics -- both paths here run the
# SAME eager model, so eager-vs-sdpa noise (the actual source of any
# legitimate cross-notebook fp difference) never enters this comparison.
# Any divergence below comes only from how this notebook's OWN hand-rolled
# loop drives the model, i.e. exactly the class of bug ("masking/position
# bug in the loop") this cell exists to catch.
#
# Small text differences after the graded answer (or in perplexity's last
# decimal places) can still occur from fp summation-order effects between
# generate()'s internal step and this loop's explicit per-step forward
# calls -- greedy argmax decoding is deterministic given identical logits,
# but the two code paths are not guaranteed to sum the same floating-point
# operations in the same order. What must NOT differ is the graded answer
# itself: if the predicted final number disagrees, that is not fp noise, it
# is a real bug in baseline_step's position_ids/attention_mask -- fix it
# before reading any timing number produced below.

_val_qa = gsm8k_qa_pairs[0]
_val_prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {_val_qa['question'].strip()}\nAnswer:"
_val_enc = tokenizer(_val_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    _ref_out = model_fp.generate(
        **_val_enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
_ref_text = tokenizer.decode(_ref_out[0][_val_enc["input_ids"].shape[1]:], skip_special_tokens=True)
_ref_text = _ref_text.split("Question:")[0]
_ref_pred = _extract_final_number(_ref_text)

_hand_result = generate_gsm8k(_val_qa["question"])
_hand_pred = _extract_final_number(_hand_result["gen_text"])

print("Gold answer:", _val_qa["gold"])
print("\n--- reference: plain model.generate() ---")
print(_ref_text.strip())
print("predicted:", _ref_pred)
print("\n--- hand-rolled: generate_gsm8k() (used for every question below) ---")
print(_hand_result["gen_text"].strip())
print("predicted:", _hand_pred)

_exact_match = _ref_text.strip() == _hand_result["gen_text"].strip()
print("\nExact text match:", _exact_match)
if not _exact_match:
    print("NOTE: texts differ -- small differences (fp summation-order noise) are expected;")
    print("what matters is whether the PREDICTED ANSWER below still agrees.")

assert _ref_pred == _hand_pred, (
    f"Predicted final numbers disagree (reference={_ref_pred}, hand-rolled={_hand_pred}) -- "
    "this points to a masking/position-id bug in baseline_step, not benign fp noise. "
    "Do not trust any latency/accuracy number from this notebook until this is fixed."
)
print("\nVALIDATION PASSED: the hand-rolled loop's predicted answer matches plain generate() on this example.")
print("(score_mc_choice_baseline() below reuses the exact same baseline_step machinery,")
print(" so this also validates the ARC-Challenge/HellaSwag likelihood-scoring path.)")


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass) -- the full-precision
# baseline.


def evaluate_gsm8k(qa_pairs, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 1000
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k(qa["question"])
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": result["full_prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k(gsm8k_qa_pairs, METHOD_NAME),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)


## ARC-Challenge

In [ ]:
# Block - ARC-Challenge loading: ALL official splits (train + validation +
# test) combined, then random sampling -- mirrors GSM8K's own loading exactly
# (GSM8K combines train+test since it has no validation split; ARC-Challenge
# has all three, so all three go in). Every row across the combined pool gets
# the same validity filtering, then seeded_subset draws up to QA_EVAL_SAMPLES
# reproducibly. Scored by teacher-forced per-choice likelihood via
# score_mc_question_* (Helper Functions, above), zero-shot, not generation.


def load_arc_challenge_items():
    arc_train = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="train",
        desc="ARC-Challenge train load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_validation = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="validation",
        desc="ARC-Challenge validation load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_test = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )
    arc_all = list(arc_train) + list(arc_validation) + list(arc_test)

    valid_items = []
    for row in arc_all:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available "
        f"(train+validation+test combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_baseline (Helper Functions),
# then reports character-length normalized MC accuracy -- matches
# large_sample_implementations' evaluate_arc_kvquant/h2o methodology exactly
# (teacher-forced per-choice likelihood, not generation). Aggregation mirrors
# GSM8K: perplexity = MEAN of per-question perplexities (from each question's
# gold choice), TTFT/TBT/latency = means over questions, peak_memory_mb = max
# over questions, average_memory_mb = mean over questions.


def evaluate_arc_baseline(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_baseline(prompt, choice_texts, gold_index)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        choices_block = "\n".join(f"{label}. {text}" for label, text in item["choices"])
        prompt_and_choices = f"{item['question']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_baseline(arc_items, METHOD_NAME),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)


## HellaSwag

In [ ]:
# Block - HellaSwag loading: ALL LABELED official splits (train +
# validation) combined, then random sampling -- mirrors GSM8K's own loading
# as closely as HellaSwag allows: HellaSwag's test split ships unlabeled
# (label == -1, no gold answer to score against), so it cannot be included
# the way GSM8K's test split is; train + validation is the full labeled pool
# available. Every row across the combined pool gets the same validity
# filtering, then seeded_subset draws up to QA_EVAL_SAMPLES reproducibly.
# Scored by teacher-forced per-choice likelihood via score_mc_question_*
# (Helper Functions, above), zero-shot, not generation.


def hellaswag_preprocess(text):
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def _hs_valid(item):
    label = str(item.get("label", "")).strip()
    endings = item.get("endings", [])
    return label.isdigit() and len(endings) >= 2 and 0 <= int(label) < len(endings)


def load_hellaswag_items():
    hs_train = robust_call(
        load_dataset, "Rowan/hellaswag", split="train",
        desc="HellaSwag train load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_validation = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )
    hs_all = list(hs_train) + list(hs_validation)

    processed_items = []
    for row in hs_all:
        if not _hs_valid(row):
            continue
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} valid examples available "
        f"(train+validation combined; test excluded -- unlabeled); "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_baseline (Helper Functions) -- the
# same machinery ARC-Challenge uses, since both are likelihood-scored MC
# datasets. Aggregation is identical to ARC-Challenge: normalized accuracy,
# perplexity = MEAN of per-question perplexities (from each example's gold
# ending), TTFT/TBT/latency = means over examples, peak_memory_mb = max over
# examples, average_memory_mb = mean over examples.


def evaluate_hellaswag_baseline(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_baseline(item["prompt"], item["choices"], item["gold_index"])

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        letters = "ABCDEFGHIJ"[:len(item["choices"])]
        choices_block = "\n".join(f"{letter}. {choice}" for letter, choice in zip(letters, item["choices"]))
        prompt_and_choices = f"{item['prompt']}\n{choices_block}"

        per_item_records.append({
            "item_index": idx,
            "prompt": prompt_and_choices,
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_baseline(hellaswag_items, METHOD_NAME),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)


## RULER


In [ ]:
# ============================================================================
# RULER settings + single-needle-in-a-haystack (NIAH) example generation.
#
# RULER (Hsieh et al., "What's the Real Context Size of Your Long-Context
# Language Models?", arXiv:2404.06654) is a SYNTHETIC long-context benchmark
# GENERATOR, not a fixed dataset -- NVIDIA's own reference implementation
# builds examples per-tokenizer/per-length/per-seed rather than shipping one
# canonical pre-built dataset (there is no single official HF dataset
# covering 4K/8K/16K/32K for an arbitrary model). This notebook generates
# RULER's flagship task -- single-needle retrieval (niah_single) -- directly,
# using THIS model's own tokenizer for exact length control, following
# NVIDIA/RULER's own documented recipe (scripts/data/synthetic/niah.py /
# constants.py): a "noise" haystack (RULER's own filler-sentence haystack
# type, not an invented approximation) with one key/value "needle" sentence
# inserted at a random depth, and RULER's own official niah prompt template
# + answer_prefix (verbatim, type_needle_v="numbers").
#
# Length buckets: 4096 / 8192 / 16384 tokens (~683/683/682 samples, 2048
# total). 32768 is DELIBERATELY EXCLUDED: eager attention (required
# throughout this notebook family) materializes the full [heads, seq, seq]
# attention-score matrix, which at 32K tokens is on the order of 60-140GB
# for a SINGLE layer -- a risk shared identically by every compression
# method here (compression shrinks the KV CACHE, not the attention-score
# computation itself, so no method here is protected from it). 16384 is
# still non-trivial (~17GB/layer at bf16) -- if it OOMs on your GPU, shrink
# RULER_LENGTH_BUCKETS below; that is a hardware ceiling, not a code bug.
#
# Grading matches RULER's own convention: does the gold value string appear
# in the model's generated text (substring/recall match), not exact-string
# equality of the whole output.

import uuid

RULER_LENGTH_BUCKETS = [4096, 8192, 16384]
RULER_MAX_NEW_TOKENS = 128  # matches RULER's own niah task config (tokens_to_generate=128)

_ruler_base, _ruler_rem = divmod(QA_EVAL_SAMPLES, len(RULER_LENGTH_BUCKETS))
RULER_SAMPLES_PER_BUCKET = {
    length: _ruler_base + (1 if i < _ruler_rem else 0)
    for i, length in enumerate(RULER_LENGTH_BUCKETS)
}

# RULER's own "noise" haystack type (NVIDIA/RULER niah.py, haystack_type=
# "noise"): a fixed pool of filler sentences, repeated to reach the target
# length.
RULER_HAYSTACK_SENTENCES = [
    "The grass is green.",
    "The sky is blue.",
    "The sun is yellow.",
    "Here we go.",
    "There and back again.",
]
RULER_SENTENCE_TOKENS = {
    s: len(tokenizer(s, add_special_tokens=False)["input_ids"])
    for s in RULER_HAYSTACK_SENTENCES
}

# RULER's own official niah prompt template + answer_prefix (verbatim, from
# NVIDIA/RULER scripts/data/synthetic/constants.py's 'niah' task, with
# type_needle_v="numbers"), plus one explicit generation instruction so the
# model is told exactly how to format its answer -- matching this notebook
# family's convention of stating the expected output format explicitly
# (the same way GSM8K's few-shot prefix states "end with exactly this
# format: #### <final number>").
_RULER_HEADER = (
    "Some special magic numbers are hidden within the following text. Make "
    "sure to memorize them. I will quiz you about the numbers afterwards. "
    "Respond with only the magic number and nothing else.\n"
)
_RULER_QUERY_TAIL = (
    "\nWhat is the special magic number for {key} mentioned in the provided text?"
    "\nAnswer: The special magic number for {key} mentioned in the provided text is"
)


def _ruler_make_needle(rng):
    """RULER's own 'uuids' key type / 'numbers' value type (both official
    RULER type_needle options) -- no external word-list dependency needed."""
    key = str(uuid.UUID(int=rng.getrandbits(128)))
    value = str(rng.randint(1000000, 9999999))  # 7-digit number, RULER's default
    return key, value


def _ruler_build_item(rng, target_tokens):
    key, value = _ruler_make_needle(rng)
    needle_sentence = f"One of the special magic numbers for {key} is: {value}."

    # Build haystack sentences until the filler alone covers target_tokens
    # (a small, roughly-fixed header/query/answer-prefix/needle overhead
    # sits on top -- RULER's own lengths are nominal/approximate too, not
    # exact byte-for-byte token counts).
    sentences = []
    token_count = 0
    while token_count < target_tokens:
        sentence = rng.choice(RULER_HAYSTACK_SENTENCES)
        sentences.append(sentence)
        token_count += RULER_SENTENCE_TOKENS[sentence]

    # Insert the needle at a random depth (0-100% of the haystack), matching
    # RULER's own DEPTHS sampling (NVIDIA/RULER niah.py).
    insert_at = rng.randint(0, len(sentences))
    sentences.insert(insert_at, needle_sentence)
    context = " ".join(sentences)

    prompt = _RULER_HEADER + context + _RULER_QUERY_TAIL.format(key=key)

    return {"prompt": prompt, "key": key, "gold_value": value, "target_tokens": target_tokens}


def build_ruler_items():
    rng = random.Random(SHARED_SEED)
    items = []
    for target_tokens in RULER_LENGTH_BUCKETS:
        n = RULER_SAMPLES_PER_BUCKET[target_tokens]
        for _ in range(n):
            items.append(_ruler_build_item(rng, target_tokens))
    return items


ruler_items = build_ruler_items()
print(
    f"RULER: generated {len(ruler_items)} single-needle items -- "
    + ", ".join(f"{RULER_SAMPLES_PER_BUCKET[l]} @ {l} tokens" for l in RULER_LENGTH_BUCKETS)
)
_ruler_preview = ruler_items[0]
print(f"\nExample prompt (first {RULER_LENGTH_BUCKETS[0]}-token item), truncated:")
print(_ruler_preview["prompt"][:400] + " ...[haystack continues]... " + _ruler_preview["prompt"][-300:])
print(f"\nGold value: {_ruler_preview['gold_value']}")


In [ ]:
# Block - RULER driver: hand-rolled prefill + greedy-decode loop, identical
# structure to generate_gsm8k/evaluate_gsm8k (baseline_prefill/baseline_step,
# same TTFT/TBT/latency definitions), scoring a single generated answer per
# item rather than choices. Perplexity is the mean NLL of the model's own
# generated tokens (no "####"-style span to isolate here, so it covers the
# whole short answer span, unlike GSM8K's marker-scoped perplexity).
#
# Memory: dense_kv_cache_bytes_for_length computes the peak ANALYTICALLY
# from n_tokens (all RULER contexts share this notebook's one dense-cache
# formula) instead of paying a second full-length prefill just to read out
# a tensor shape n_tokens already determines -- GSM8K/ARC/HellaSwag's
# untimed-second-pass approach is affordable at their prompt lengths, but
# doubling an already-expensive O(seq^2) prefill at up to 16K tokens for a
# shape lookup is not.
#
# CSV note: full_prompt is NOT written per row here (unlike GSM8K) -- a
# RULER prompt is up to ~16K tokens of synthetic haystack text, and writing
# that out 2048 times would bloat the CSV for no analytical benefit;
# length_bucket identifies which of the three context lengths each row is.


def dense_kv_cache_bytes_for_length(n_tokens):
    """Analytic dense KV cache size for a sequence of n_tokens (bf16, this
    notebook's model dtype) -- avoids a redundant full-length forward pass
    purely to measure a tensor shape."""
    n_kv_heads = model_fp.config.num_key_value_heads
    head_dim = model_fp.config.hidden_size // model_fp.config.num_attention_heads
    n_layers = model_fp.config.num_hidden_layers
    bytes_per_element = next(model_fp.parameters()).element_size()
    return int(n_tokens * n_layers * 2 * n_kv_heads * head_dim * bytes_per_element)


@torch.no_grad()
def generate_ruler_baseline(prompt):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    sync_if_cuda()
    gen_start = time.perf_counter()

    last_logits, pkv = baseline_prefill(enc["input_ids"])
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(RULER_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):  # early stop once the answer line ends
            break

        if step == RULER_MAX_NEW_TOKENS - 1:
            break  # no wasted forward for a token we would never use

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = baseline_step(token_tensor, abs_pos, pkv)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated
    peak_bytes = dense_kv_cache_bytes_for_length(total_tokens)

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": total_tokens, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_ruler_baseline(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"RULER | {method_label}")):
        result = generate_ruler_baseline(item["prompt"])
        is_correct = item["gold_value"] in result["gen_text"]
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- RULER | {method_label} | question {q_idx} ({item['target_tokens']} tokens) preview ---")
            print(f"Gold value: {item['gold_value']}")
            print(f"Generated:  {result['gen_text'].strip()!r}")
            print(f"Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "length_bucket": item["target_tokens"],
            "full_prompt": item["prompt"],
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": item["gold_value"],
            "correct": int(is_correct),
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_ruler_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question RULER rows to {_per_prompt_path}")

    return {
        "dataset": "RULER",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


ruler_results = [
    evaluate_ruler_baseline(ruler_items, METHOD_NAME),
]
ruler_results_df = pd.DataFrame(ruler_results)
display(ruler_results_df)


## SQuAD1.1


In [ ]:
# ============================================================================
# SQuAD1.1 (rajpurkar/squad -- the standard v1.1 release, every question has
# an answer in its passage, unlike squad_v2's unanswerable questions):
# extractive reading comprehension. Loading combines train + validation
# (SQuAD ships no public-label test split), matching the GSM8K/ARC/HellaSwag
# convention of pooling every official labeled split before a seeded sample.
# Zero-shot, generation-based, graded with the OFFICIAL SQuAD metrics --
# Exact Match and F1 -- against the best of however many gold reference
# answers a question has (validation questions often carry more than one
# accepted answer; train questions usually carry exactly one).
# ============================================================================

import re
import string
from collections import Counter

SQUAD_MAX_NEW_TOKENS = 48  # SQuAD gold answers are short spans (usually 1-5 words)

SQUAD_HEADER = (
    "Answer the question based on the passage below. Respond with a short "
    "phrase copied directly from the passage that answers the question, and "
    "nothing else.\n\n"
)


def format_squad_prompt(item):
    return (
        SQUAD_HEADER
        + f"Passage: {item['context'].strip()}\n"
        + f"Question: {item['question'].strip()}\n"
        + "Answer:"
    )


def load_squad_items():
    squad_train = robust_call(
        load_dataset, "rajpurkar/squad", split="train",
        desc="SQuAD1.1 train load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_validation = robust_call(
        load_dataset, "rajpurkar/squad", split="validation",
        desc="SQuAD1.1 validation load", on_retry=lambda: clear_hf_dataset_cache("squad"),
    )
    squad_all = list(squad_train) + list(squad_validation)

    valid_items = []
    for row in squad_all:
        texts = row.get("answers", {}).get("text", [])
        gold_answers = [str(t).strip() for t in texts if str(t).strip()]
        if not gold_answers or not str(row.get("context", "")).strip() or not str(row.get("question", "")).strip():
            continue
        valid_items.append({
            "context": row["context"],
            "question": row["question"],
            "gold_answers": gold_answers,
        })

    selected_items, selected_indices = seeded_subset(valid_items, QA_EVAL_SAMPLES, SHARED_SEED)
    print(
        f"SQuAD1.1: {len(valid_items)} valid questions available "
        f"(train+validation combined); selected {len(selected_items)} "
        f"random questions (requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("SQuAD1.1 selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


squad_items = load_squad_items()


# ---------------------------------------------------------------------------
# Official SQuAD1.1 scoring (Rajpurkar et al. 2016's own normalize_answer /
# exact_match_score / f1_score, reproduced verbatim): lowercase, drop
# punctuation, drop articles (a/an/the), collapse whitespace, then compare.
# Both metrics take the MAX over every gold reference answer available for
# that question.
# ---------------------------------------------------------------------------

def _squad_normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = " ".join(text.split())
    return text


def squad_exact_match(prediction, gold_answers):
    norm_pred = _squad_normalize(prediction)
    return max(int(norm_pred == _squad_normalize(g)) for g in gold_answers)


def squad_f1(prediction, gold_answers):
    pred_tokens = _squad_normalize(prediction).split()
    best = 0.0
    for g in gold_answers:
        gold_tokens = _squad_normalize(g).split()
        if len(pred_tokens) == 0 or len(gold_tokens) == 0:
            best = max(best, float(pred_tokens == gold_tokens))
            continue
        common = Counter(pred_tokens) & Counter(gold_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = num_same / len(pred_tokens)
        recall = num_same / len(gold_tokens)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


In [ ]:
# Block - SQuAD1.1 driver: hand-rolled prefill + greedy-decode loop,
# identical structure to generate_gsm8k/evaluate_gsm8k (baseline_prefill/
# baseline_step, same TTFT/TBT/latency definitions, same
# dense_kv_cache_bytes_for_length analytic memory helper the RULER section
# also defines -- redefined here too (harmless if both run; Python just
# keeps the latest def) so this cell works standalone even if RULER's
# cells were never run in this session), scoring a short generated answer
# per question against SQuAD's official Exact Match / F1 metrics
# (best-of however many gold reference answers a question has).


def dense_kv_cache_bytes_for_length(n_tokens):
    """Analytic dense KV cache size for a sequence of n_tokens (bf16, this
    notebook's model dtype) -- avoids a redundant full-length forward pass
    purely to measure a tensor shape. Duplicated from the RULER section so
    this cell has no dependency on RULER having been run first."""
    n_kv_heads = model_fp.config.num_key_value_heads
    head_dim = model_fp.config.hidden_size // model_fp.config.num_attention_heads
    n_layers = model_fp.config.num_hidden_layers
    bytes_per_element = next(model_fp.parameters()).element_size()
    return int(n_tokens * n_layers * 2 * n_kv_heads * head_dim * bytes_per_element)


@torch.no_grad()
def generate_squad_baseline(prompt):
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    sync_if_cuda()
    gen_start = time.perf_counter()

    last_logits, pkv = baseline_prefill(enc["input_ids"])
    next_id = int(last_logits.argmax(dim=-1)[0].item())
    sync_if_cuda()
    ttft_sec = time.perf_counter() - gen_start

    generated_ids = []
    generated_logits = []
    n_forward_tokens = 1

    for step in range(SQUAD_MAX_NEW_TOKENS):
        if next_id == tokenizer.eos_token_id:
            break
        generated_ids.append(next_id)
        generated_logits.append(last_logits.detach())
        if "\n" in tokenizer.decode([next_id]):
            break

        if step == SQUAD_MAX_NEW_TOKENS - 1:
            break

        token_tensor = torch.tensor([[next_id]], dtype=torch.long, device=device)
        abs_pos = prompt_len + step
        last_logits, pkv = baseline_step(token_tensor, abs_pos, pkv)
        next_id = int(last_logits.argmax(dim=-1)[0].item())
        n_forward_tokens += 1

    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    if generated_ids:
        nll_sum = 0.0
        for tok_id, logits in zip(generated_ids, generated_logits):
            log_probs = torch.log_softmax(logits[0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
        perplexity = math.exp(min(nll_sum / len(generated_ids), 50.0))
    else:
        perplexity = None

    gen_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    n_generated = n_forward_tokens
    if len(generated_ids) == 0:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated
    peak_bytes = dense_kv_cache_bytes_for_length(total_tokens)

    return {
        "prefill_tokens": prompt_len, "generated_tokens": len(generated_ids), "gen_text": gen_text,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec, "total_latency_sec": total_latency_sec,
        "total_tokens": total_tokens, "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


def evaluate_squad_baseline(items, method_label):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values, f1_values = [], [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, item in enumerate(tqdm(items, desc=f"SQuAD1.1 | {method_label}")):
        prompt = format_squad_prompt(item)
        result = generate_squad_baseline(prompt)
        prediction = result["gen_text"].strip()
        em = squad_exact_match(prediction, item["gold_answers"])
        f1 = squad_f1(prediction, item["gold_answers"])
        correct += em
        total += 1
        f1_values.append(f1)

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- SQuAD1.1 | {method_label} | question {q_idx} preview ---")
            print(f"Question:     {item['question']}")
            print(f"Gold answers: {item['gold_answers']}")
            print(f"Generated:    {prediction!r}")
            print(f"EM: {em} | F1: {f1:.3f}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        if result["perplexity"] is not None:
            ppl_values.append(result["perplexity"])

        per_question_records.append({
            "question_index": q_idx,
            "full_prompt": prompt,
            "prefill_tokens": result["prefill_tokens"],
            "generated_output": result["gen_text"],
            "generated_tokens": result["generated_tokens"],
            "gold_answer": " / ".join(item["gold_answers"]),
            "predicted_answer": prediction,
            "correct": em,
            "f1": f1,
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_mb": result["peak_memory_bytes"] / 1024**2,
        })

    accuracy = correct / max(total, 1)
    avg_f1 = sum(f1_values) / len(f1_values) if f1_values else float("nan")
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_squad_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question SQuAD1.1 rows to {_per_prompt_path}")

    return {
        "dataset": "SQuAD1.1",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,  # Exact Match
        "avg_f1": avg_f1,      # official SQuAD F1 -- extra column, dropped by the final
                                # Save Results subset but visible here and in this dataset's
                                # own per-prompt CSV / summary display
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_mb": max(peak_mem_values) / 1024**2 if peak_mem_values else 0.0,
        "average_memory_mb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**2) if peak_mem_values else 0.0,
    }


squad_results = [
    evaluate_squad_baseline(squad_items, METHOD_NAME),
]
squad_results_df = pd.DataFrame(squad_results)
display(squad_results_df)


## Save Results

In [ ]:
# Block - Save combined results (GSM8K + ARC-Challenge + HellaSwag) to CSV.
# One row per dataset for this single full-precision baseline method. The
# per-question CSVs are written separately by each dataset's evaluate step.
#
# Cross-method comparison reminders:
#   - This notebook now runs eager attention with a hand-rolled
#     prefill/decode loop (see the model-loading and machinery cells
#     above), matching H2O's and KVQuant's loop structure and attention
#     kernel exactly -- there is no longer an "eager tax" confound between
#     this notebook and the other two; any remaining latency gap is
#     attributable to H2O's eviction / KVQuant's quantize-pack-dequantize
#     overhead, not to SDPA vs. eager.
#   - For cross-method comparison, prefer average_memory_mb over
#     peak_memory_mb as the primary memory axis. peak_memory_mb is the max
#     over 2,048 questions and is therefore dominated by whichever few
#     questions happen to generate the most tokens (up to
#     GSM8K_MAX_NEW_TOKENS) -- a property of which SAMPLES landed in the
#     seeded subset, not of the method being evaluated. On GSM8K in
#     particular, generation length varies a lot question-to-question (some
#     finish in a few dozen tokens, some run to the cap), so peak and
#     average are expected to diverge meaningfully here; ARC/HellaSwag are no
#     longer generated at all (likelihood-scored over each answer choice), so
#     their peak/average gap instead reflects how much a question's choice
#     texts vary in token length -- typically much smaller than GSM8K's gap,
#     but worth confirming empirically once this notebook has been run.
#   - Confirm the GPU name printed in Block 2 matches every other notebook's
#     run before trusting any timing/memory comparison.

# Robust to partial runs: only concatenates whichever of gsm8k_results_df /
# arc_results_df / hellaswag_results_df / ruler_results_df actually exist in
# this session, so you can run just a subset of datasets' cells without this
# cell crashing on a NameError for a dataframe you never created -- RULER in
# particular may not always finish (16K-token eager attention is memory-
# heavy), so this matters more here than it did before RULER existed.
_result_df_names = ["gsm8k_results_df", "arc_results_df", "hellaswag_results_df", "ruler_results_df", "squad_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

all_results_df = pd.concat(_available_dfs, ignore_index=True)

results_df = all_results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_mb", "average_memory_mb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_baseline_full_precision_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")


In [ ]:
from google.colab import runtime
runtime.unassign()